# Diabetes Prediction Model (NHANES 2017–2018)

**Goal:** a screening model that flags adults likely to have diabetes using **only physical exam measurements** (body measures, blood pressure, pulse) plus age and sex. **No blood test of any kind** is needed to use it.

**Pipeline (one section per step):**

Each step lists the data it uses and its sub-steps. 🔁 marks a sub-step that **learns from data** (for example, a median or a mean). Those go inside a scikit-learn `Pipeline`, so they are refit on the training part of every CV fold and never see validation or test rows.

1. **Feature identification** ← *this section* · *all data*
   - 1.1–1.4 Load and join the raw tables
   - 1.5 Define the target
   - 1.6–1.7 Apply the cohort filters
   - 1.8–1.11 Feature catalog, leakage guard, final table preview
2. **Feature table** · *all data*
   - 2.1 Fix storage glitches (tiny floats like `5.4e-79` that are really 0)
   - 2.2 Diastolic BP of 0 means "not measurable" → missing
   - 2.3 Recode skipped questions (`taking_bp_meds`: blank means "no")
   - 2.4 Fix data types (e.g. `gender` is a category stored as 1.0/2.0)
   - 2.5 Sanity checks: one row per person, target is 0/1, values physically possible
   - 2.6 **Stratified 80/20 train/test split**; the test set is sealed from here on
   - 2.7 Check the split
3. **Exploratory data analysis (EDA)** · *train only*
   - 3.1 Target balance
   - 3.2 Missing values per feature
   - 3.3 Distributions and outliers
   - 3.4 Each feature vs the target
   - 3.5 Correlations and redundant features (e.g. BMI, weight and waist)
   - 3.6 Effect of blood pressure medication on readings
4. **Feature preparation and engineering** · *rules decided on train*
   - 4.1 Outlier handling: clip or remove impossible values
   - 4.2 Derived features: mean systolic and diastolic BP, waist-to-height, waist-to-hip, leg-to-height ratios
   - 4.3 🔁 **Imputing** missing values (e.g. median) plus missing-value flags
   - 4.4 🔁 **Encoding** categorical columns (e.g. one-hot for `gender`)
   - 4.5 🔁 **Normalization**: scale numeric columns to a common range (needed for logistic regression, not for tree models)
   - 4.6 Assemble the preprocessing `Pipeline` (`ColumnTransformer`)
5. **Base model** · *train, stratified 5-fold CV*
   - 5.1 Dummy baseline that always predicts "not diabetic" (the accuracy trap)
   - 5.2 Logistic regression baseline
6. **Model fitting and tuning** · *train, stratified 5-fold CV*
   - 6.1 Candidate models: logistic regression, random forest, gradient boosting
   - 6.2 Class imbalance handling (e.g. `class_weight="balanced"`)
   - 6.3 Hyperparameter search (`GridSearchCV`)
7. **Metrics** · *train, CV results*
   - 7.1 Ranking: ROC-AUC, average precision (PR-AUC)
   - 7.2 Classification: confusion matrix, recall, precision, F1
   - 7.3 Probability quality: Brier score, calibration curve
8. **Recalibration and optimization** · *train, CV folds only*
   - 8.1 🔁 Probability calibration (`CalibratedClassifierCV`)
   - 8.2 Threshold tuning for screening (favor recall: miss as few diabetics as possible)
   - 8.3 Feature ablation: does each feature group actually help?
   - 8.4 Experiment: correcting blood pressure for medication
9. **Final evaluation** · *test, opened once*
   - 9.1 Refit the chosen pipeline on the full training set
   - 9.2 Score the test set once with the chosen threshold
   - 9.3 Slice checks: performance by sex and age group
   - 9.4 Save the final model

**Ground rules:**
- **The test set is a sealed envelope.** It is split off before EDA, so no decision is ever based on it, and it is used exactly once, in Step 9.
- **Stratified splits** keep the diabetic rate (about 16%) the same in train, test and every CV fold.
- **The tables are rebuilt from the raw data on every run**, not saved and reloaded, so a later step can never use a stale copy.

Run the cells top to bottom. Each cell does one job, described in the markdown cell above it.

## Step 1 — Feature identification

Three questions, answered in this order:
1. **Target:** what exactly are we predicting?
2. **Cohort:** whose data do we learn from?
3. **Features:** which columns may the model see?

Two rules decide the features:
- **Scope rule:** only physical exam measurements, plus age and sex.
- **Leakage rule:** a feature is allowed only if it would be known **before** we know the answer, at the moment we want to predict.

Blood tests (HbA1c, questionnaire answers) are still used to **define the target**; they are just never shown to the model.

### 1.1 Imports, paths and constants

Loads the libraries, points to the `Data/` folder, and sets every number the later cells rely on in one place, so nothing is hard-coded further down.

NHANES stores answers as numeric codes. For yes/no questions, `1` = yes, `2` = no, `7` = refused, `9` = don't know.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../Data")

MIN_AGE = 20                 # adults only
HBA1C_DIABETES_CUTOFF = 6.5  # %, the clinical threshold for diabetes
YES = 1                      # NHANES code for "yes"

TEST_SIZE = 0.2              # share of the cohort held out as the test set
RANDOM_STATE = 42            # fixed seed, so the split is the same on every run

pd.set_option("display.max_columns", 50)

### 1.2 Load the raw tables

Reads the six NHANES tables this project needs into a dictionary called `raw`, keyed by a short name, and prints the size of each one. Nothing is changed or joined yet.

- `demographics`, `body`, `blood_pressure`: sources of the features (and the pregnancy filter)
- `questionnaire`, `hba1c`: used only to build the target and the cohort filters
- `bp_questionnaire`: source of `taking_bp_meds`, kept for EDA only (see 1.8)

In [2]:
TABLE_FILES = {
    "demographics": "demographics.csv",
    "body": "body_measures.csv",
    "blood_pressure": "blood_pressure.csv",
    "questionnaire": "diabetes_questionnaire.csv",
    "hba1c": "hba1c.csv",
    "bp_questionnaire": "bp_cholesterol_questionnaire.csv",
}

raw = {name: pd.read_csv(DATA_DIR / file) for name, file in TABLE_FILES.items()}

pd.DataFrame(
    {name: {"rows": len(df), "columns": df.shape[1]} for name, df in raw.items()}
).T

,rows,columns
demographics,9254,46
body,8704,21
blood_pressure,8704,21
questionnaire,8897,54
hba1c,6401,2
bp_questionnaire,6161,11


### 1.3 Check the join key

Every table identifies a person by `person_id`. Before joining, this cell confirms each table has **at most one row per person**. A duplicated id would silently copy that person's rows during the join.

In [3]:
for name, df in raw.items():
    assert df["person_id"].is_unique, f"{name} has duplicate person_id values"

print("All tables have one row per person_id.")

All tables have one row per person_id.


### 1.4 Join the tables into one wide table

Starts from `demographics` (everyone in the survey) and **left-joins** every other table on `person_id`. A left join keeps every person, even if they have no row in a table; their missing measurements become `NaN`.

In [4]:
merged = raw["demographics"]
for name, df in raw.items():
    if name != "demographics":
        merged = merged.merge(df, on="person_id", how="left")
merged = merged.copy()  # repeated joins leave pandas' internal storage fragmented; copying tidies it

print(f"{merged.shape[0]:,} people x {merged.shape[1]} columns")

9,254 people x 150 columns


### 1.5 Define the target

A person counts as **diabetic** (`diabetes = 1`) if **either**:
- a doctor has told them they have diabetes (`told_diabetes == 1`), **or**
- their HbA1c is at or above 6.5%, which catches people with **undiagnosed** diabetes.

Answers of *borderline* (`3`) or *don't know* (`9`) count as diabetic only if their HbA1c is 6.5% or higher.

Because these columns define the answer, they can **never** be features.

In [5]:
def make_target(df):
    told = df["told_diabetes"] == YES
    high_a1c = df["hba1c_percent"] >= HBA1C_DIABETES_CUTOFF
    return (told | high_a1c).astype(int)


merged["diabetes"] = make_target(merged)

### 1.6 Apply the cohort filters

Each filter is a named rule. They are applied in order, and the table printed afterward (the **cohort funnel**) shows how many people and diabetic cases each rule leaves. This makes every exclusion visible and easy to question.

| Filter | Why |
|---|---|
| Adult (20+) | Diabetes in children is mostly type 1, a different disease |
| Has an HbA1c result | Without it the target can't be checked for undiagnosed diabetes |
| Not pregnant | Pregnancy lowers HbA1c and changes weight and blood pressure |
| Not taking insulin | Insulin users have advanced, already-treated diabetes; the screening population isn't on insulin |

In [6]:
COHORT_FILTERS = {
    f"age >= {MIN_AGE}": lambda df: df["age_years"] >= MIN_AGE,
    "has HbA1c result": lambda df: df["hba1c_percent"].notna(),
    "not pregnant": lambda df: df["pregnant_at_exam"] != YES,
    "not taking insulin": lambda df: df["taking_insulin"] != YES,
}

cohort = merged
funnel = [{"step": "all survey participants", "people": len(cohort), "diabetic": cohort["diabetes"].sum()}]
for step, rule in COHORT_FILTERS.items():
    cohort = cohort[rule(cohort)]
    funnel.append({"step": step, "people": len(cohort), "diabetic": cohort["diabetes"].sum()})

funnel = pd.DataFrame(funnel)
funnel["diabetic_rate"] = (funnel["diabetic"] / funnel["people"]).round(3)
funnel

,step,people,diabetic,diabetic_rate
0,all survey participants,9254,1085,0.117
1,age >= 20,5569,1066,0.191
2,has HbA1c result,5018,994,0.198
3,not pregnant,4971,994,0.200
4,not taking insulin,4742,766,0.162


### 1.7 Break down the diabetic cases

Splits the cohort's diabetic cases into **diagnosed** (a doctor told them) and **undiagnosed** (HbA1c ≥ 6.5% but never told). The undiagnosed group is who a screening model exists to find.

In [7]:
told = cohort["told_diabetes"] == YES
undiagnosed = (cohort["hba1c_percent"] >= HBA1C_DIABETES_CUTOFF) & ~told

pd.Series({
    "diagnosed": told.sum(),
    "undiagnosed": undiagnosed.sum(),
    "not diabetic": (cohort["diabetes"] == 0).sum(),
})

diagnosed        577
undiagnosed      189
not diabetic    3976
dtype: int64

### 1.8 Feature catalog: every candidate column and its decision

Lists the candidate columns with a **group**, a **decision** and the **reason**. This table is the written record of Step 1. There are three decisions:
- `use`: a model feature
- `eda`: carried along in the table for analysis, but **never** shown to the model
- `drop`: not carried at all

`taking_bp_meds` is `eda` for now. Over half of the diabetic cases take blood pressure medicine, which lowers their readings, and EDA (3.6) will measure that effect. To try it as a feature later, change its decision to `use`.

Blood pressure keeps readings 1–3 for now. Combining them into one average is a feature engineering job (Step 4). Reading 4 is dropped: NHANES only takes it when an earlier reading failed, so it is missing for over 90% of people.

In [8]:
catalog = pd.DataFrame(
    [
        # column, group, decision, reason
        ("age_years", "age and sex", "use", "Major diabetes risk factor; always known"),
        ("gender", "age and sex", "use", "Risk and body shape differ by sex; always known"),
        ("bmi", "body", "use", "Strongest known risk factor"),
        ("waist_cm", "body", "use", "Belly fat; adds signal beyond BMI"),
        ("hip_cm", "body", "use", "Gives waist-to-hip ratio in Step 4"),
        ("weight_kg", "body", "use", "Raw body size"),
        ("height_cm", "body", "use", "Raw body size; gives waist-to-height ratio in Step 4"),
        ("arm_circumference_cm", "body", "use", "Muscle and fat on the arm"),
        ("upper_leg_length_cm", "body", "use", "Shorter legs relative to height are linked to diabetes risk"),
        ("upper_arm_length_cm", "body", "use", "Body proportion measure"),
        ("systolic_bp_1", "blood pressure", "use", "Reading 1 of 3; averaged in Step 4"),
        ("systolic_bp_2", "blood pressure", "use", "Reading 2 of 3"),
        ("systolic_bp_3", "blood pressure", "use", "Reading 3 of 3"),
        ("diastolic_bp_1", "blood pressure", "use", "Reading 1 of 3; averaged in Step 4"),
        ("diastolic_bp_2", "blood pressure", "use", "Reading 2 of 3"),
        ("diastolic_bp_3", "blood pressure", "use", "Reading 3 of 3"),
        ("pulse_60s", "pulse", "use", "Higher resting heart rate is linked to diabetes"),
        ("taking_bp_meds", "medication", "eda", "BP medicine lowers readings; questionnaire answer, so not a feature for now"),
        ("systolic_bp_4", "blood pressure", "drop", "Only taken when an earlier reading failed; mostly missing"),
        ("diastolic_bp_4", "blood pressure", "drop", "Only taken when an earlier reading failed; mostly missing"),
        ("recumbent_length_cm", "body", "drop", "Measured only for infants"),
        ("head_circumference_cm", "body", "drop", "Measured only for infants"),
        ("hba1c_percent", "target-defining", "drop", "Defines the target (direct leakage)"),
        ("told_diabetes", "target-defining", "drop", "Defines the target (direct leakage)"),
        ("taking_insulin", "cohort filter", "drop", "Used to filter the cohort; constant afterward"),
        ("pregnant_at_exam", "cohort filter", "drop", "Used to filter the cohort"),
        ("taking_diabetes_pills", "diabetes follow-up", "drop", "Only diagnosed people take them (treatment leakage)"),
        ("diabetic_retinopathy", "diabetes follow-up", "drop", "A complication of diabetes, so it comes after the answer"),
        ("race_ethnicity_with_asian", "demographics", "drop", "Not a physical measurement (out of scope)"),
        ("education_adult", "demographics", "drop", "Not a physical measurement (out of scope)"),
        ("income_to_poverty_ratio", "demographics", "drop", "Not a physical measurement (out of scope)"),
        ("design_psu", "survey design", "drop", "Survey sampling detail, not a property of the person"),
        ("exam_weight", "survey design", "drop", "Survey weight, not a property of the person"),
    ],
    columns=["column", "group", "decision", "reason"],
)

catalog

,column,group,decision,reason
0,age_years,age and sex,use,Major diabetes risk factor; always known
1,gender,age and sex,use,Risk and body shape differ by sex; always known
2,bmi,body,use,Strongest known risk factor
3,waist_cm,body,use,Belly fat; adds signal beyond BMI
4,hip_cm,body,use,Gives waist-to-hip ratio in Step 4
5,weight_kg,body,use,Raw body size
6,height_cm,body,use,Raw body size; gives waist-to-height ratio in ...
7,arm_circumference_cm,body,use,Muscle and fat on the arm
8,upper_leg_length_cm,body,use,Shorter legs relative to height are linked to ...
9,upper_arm_length_cm,body,use,Body proportion measure


### 1.9 Collect the selected features and the target

Turns the catalog into the two names the rest of the notebook uses:
- `FEATURES`: the columns marked `use`
- `EDA_ONLY`: the columns marked `eda`
- `TARGET`: the column the model predicts

In [9]:
FEATURES = catalog.loc[catalog["decision"] == "use", "column"].tolist()
EDA_ONLY = catalog.loc[catalog["decision"] == "eda", "column"].tolist()
TARGET = "diabetes"

print(f"{len(FEATURES)} features selected, {len(EDA_ONLY)} EDA-only column(s): {EDA_ONLY}")
catalog[catalog["decision"] == "use"].groupby("group", sort=False)["column"].apply(list)

17 features selected, 1 EDA-only column(s): ['taking_bp_meds']


group
age and sex                                     [age_years, gender]
body              [bmi, waist_cm, hip_cm, weight_kg, height_cm, ...
blood pressure    [systolic_bp_1, systolic_bp_2, systolic_bp_3, ...
pulse                                                   [pulse_60s]
Name: column, dtype: object

### 1.10 Leakage guard

A safety check that fails loudly if a forbidden column ever ends up in `FEATURES`, for example after someone edits the catalog by mistake. Forbidden means marked `drop` or `eda`, or the target itself. It also confirms every selected column really exists in the data.

In [10]:
FORBIDDEN = set(catalog.loc[catalog["decision"].isin(["drop", "eda"]), "column"]) | {TARGET}

leaks = FORBIDDEN & set(FEATURES)
assert not leaks, f"Leaking columns in FEATURES: {leaks}"

missing = set(FEATURES + EDA_ONLY) - set(cohort.columns)
assert not missing, f"Columns not found in the data: {missing}"

print("Leakage guard passed: no forbidden columns, all features exist.")

Leakage guard passed: no forbidden columns, all features exist.


### 1.11 Preview the final table

Builds `final_df`: one row per person in the cohort, with `person_id`, the selected `FEATURES`, the `EDA_ONLY` columns and the `TARGET` column, and nothing else. Shows the first 5 rows as a last check that Step 1 produced what we expect.

In [11]:
final_df = cohort[["person_id"] + FEATURES + EDA_ONLY + [TARGET]].reset_index(drop=True)

print(f"{final_df.shape[0]:,} people x {final_df.shape[1]} columns")
final_df.head(5)

4,742 people x 20 columns


,person_id,age_years,gender,bmi,waist_cm,hip_cm,weight_kg,height_cm,arm_circumference_cm,upper_leg_length_cm,upper_arm_length_cm,systolic_bp_1,systolic_bp_2,systolic_bp_3,diastolic_bp_1,diastolic_bp_2,diastolic_bp_3,pulse_60s,taking_bp_meds,diabetes
0,93705,66.0,2.0,31.7,101.8,110.0,79.5,158.3,32.0,37.0,36.0,NaN,NaN,202.0,NaN,NaN,62.0,52.0,1.0,0
1,93708,66.0,2.0,23.7,88.2,90.1,53.5,150.2,27.4,31.8,30.6,NaN,138.0,144.0,NaN,78.0,76.0,68.0,1.0,0
2,93709,75.0,2.0,38.9,113.0,131.3,88.8,151.1,38.1,31.9,36.0,120.0,118.0,118.0,66.0,66.0,68.0,74.0,1.0,0
3,93711,56.0,1.0,21.3,86.6,90.7,62.1,170.6,30.8,40.1,37.2,108.0,94.0,102.0,68.0,68.0,64.0,62.0,NaN,0
4,93713,67.0,1.0,23.5,99.7,91.0,74.9,178.6,30.6,39.1,41.4,104.0,104.0,106.0,70.0,74.0,72.0,90.0,NaN,0


### Step 1 summary

- **Target:** diagnosed diabetes **or** HbA1c ≥ 6.5%.
- **Cohort:** adults (20+) with an HbA1c result, not pregnant and not taking insulin. See the funnel in 1.6.
- **Features:** age, sex, body measures, three blood pressure readings and pulse. See 1.9. No blood test is needed to use the model.
- **EDA-only:** `taking_bp_meds`, kept in the table to study how medication affects blood pressure, but blocked from the model by the leakage guard.
- **Missing values are kept for now:** some people lack a body measure or a blood pressure reading. How to handle them is decided in EDA and feature preparation.
- **Excluded:** everything that defines the target, all blood tests, anything that only exists *because* someone was diagnosed, and non-physical demographics.

**Next, Step 2 (feature table):** clean `final_df` in memory (data types, NHANES codes, sanity checks), then split it into stratified train and test sets.

## Step 2 — Feature table

Turns `final_df` into a clean table, `feature_table`, then splits it into a training set and a sealed test set.

Every cleaning rule here looks at **one row at a time** (e.g. "a diastolic of 0 means missing"). Nothing is computed from the data as a whole, like a median or a mean, so it is safe to apply these rules **before** the split. Anything that learns from the data waits for Step 4, after the split.

### 2.1 Fix storage glitches

Converting NHANES files from SAS format to CSV stores an exact **0** as a tiny number such as `5.4e-79`. This cell turns every such value back into 0 and shows how many it fixed in each column.

It works on a copy, `feature_table`, so `final_df` from Step 1 stays untouched.

In [12]:
feature_table = final_df.copy()

numeric_cols = feature_table.select_dtypes("number").columns
is_glitch = (feature_table[numeric_cols].abs() < 1e-10) & (feature_table[numeric_cols] != 0)
feature_table[numeric_cols] = feature_table[numeric_cols].mask(is_glitch, 0.0)

glitches_fixed = is_glitch.sum()
glitches_fixed[glitches_fixed > 0]

diastolic_bp_1    29
diastolic_bp_2    21
diastolic_bp_3    15
dtype: int64

### 2.2 Diastolic BP of 0 means "not measurable"

NHANES records a diastolic reading of 0 when the sound that marks it can't be heard. Nobody has a diastolic pressure of 0, so the true value is **unknown**. This cell turns those zeros into `NaN` (missing) and shows how many it changed in each reading.

In [13]:
SYSTOLIC_COLS = ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
DIASTOLIC_COLS = ["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]

zero_diastolic = (feature_table[DIASTOLIC_COLS] == 0).sum()
feature_table[DIASTOLIC_COLS] = feature_table[DIASTOLIC_COLS].replace(0, float("nan"))

zero_diastolic

diastolic_bp_1    29
diastolic_bp_2    21
diastolic_bp_3    15
dtype: int64

### 2.3 Recode skipped questions

NHANES only asks "Are you now taking medicine for high blood pressure?" to people who were told to take it. Everyone else was skipped, so their blank answer means **no**. This cell turns `taking_bp_meds` into a plain 0/1 column: `1` = yes, `0` = no (answered no, or never asked).

In [14]:
feature_table["taking_bp_meds"] = (feature_table["taking_bp_meds"] == YES).astype(int)

feature_table["taking_bp_meds"].value_counts()

taking_bp_meds
0    3395
1    1347
Name: count, dtype: int64

### 2.4 Fix data types

- **`gender`:** stored as 1.0 / 2.0, which a model would read as "female = 2 × male". This cell turns it into a category with readable labels. Turning it into model-ready numbers is the encoding step (4.4).
- **`age_years`:** stored as a float but always a whole number, so it becomes an integer. Note that NHANES records everyone aged 80 or older as **80**.

The output lists every column's final type.

In [15]:
GENDER_LABELS = {1: "male", 2: "female"}

feature_table["gender"] = feature_table["gender"].map(GENDER_LABELS).astype("category")
feature_table["age_years"] = feature_table["age_years"].astype(int)

feature_table.dtypes

person_id                  int64
age_years                  int64
gender                  category
bmi                      float64
waist_cm                 float64
hip_cm                   float64
weight_kg                float64
height_cm                float64
arm_circumference_cm     float64
upper_leg_length_cm      float64
upper_arm_length_cm      float64
systolic_bp_1            float64
systolic_bp_2            float64
systolic_bp_3            float64
diastolic_bp_1           float64
diastolic_bp_2           float64
diastolic_bp_3           float64
pulse_60s                float64
taking_bp_meds             int64
diabetes                   int64
dtype: object

### 2.5 Sanity checks

Assertions that stop the notebook if the table breaks a basic rule:
- one row per person
- the target only contains 0 and 1
- every person has a gender label (no unmapped codes)
- no measurement is zero or negative
- in each reading, systolic is higher than diastolic

Missing values are allowed here; they are handled in Step 4.

In [16]:
assert feature_table["person_id"].is_unique, "duplicate person_id"
assert set(feature_table[TARGET].unique()) == {0, 1}, "target is not 0/1"
assert feature_table["gender"].notna().all(), "unmapped gender code"

measurements = feature_table[FEATURES].select_dtypes("number")
assert not (measurements <= 0).any().any(), "zero or negative measurement"

for sys_col, dia_col in zip(SYSTOLIC_COLS, DIASTOLIC_COLS):
    bad = feature_table[sys_col] <= feature_table[dia_col]
    assert not bad.any(), f"{bad.sum()} rows where {sys_col} <= {dia_col}"

print(f"All sanity checks passed: {feature_table.shape[0]:,} people x {feature_table.shape[1]} columns")

All sanity checks passed: 4,742 people x 20 columns


### 2.6 Stratified train/test split

Splits `feature_table` into:
- `train_df` (80%): used for everything in Steps 3–8
- `test_df` (20%): the **sealed envelope**, opened only once in Step 9

`stratify` splits the diabetic and non-diabetic groups separately, so both sets keep the same diabetic rate. `random_state` fixes the shuffle, so the split is identical on every run.

In [17]:
train_df, test_df = train_test_split(
    feature_table,
    test_size=TEST_SIZE,
    stratify=feature_table[TARGET],
    random_state=RANDOM_STATE,
)

print(f"train: {len(train_df):,} people | test: {len(test_df):,} people")

train: 3,793 people | test: 949 people


### 2.7 Check the split

Confirms the split did its job:
- **No overlap:** nobody appears in both sets.
- **Same mix:** the diabetic rate matches across the full table, train and test (guaranteed by stratifying), and the share of women is close (left to chance, so it should only be roughly equal).

In [18]:
assert set(train_df["person_id"]).isdisjoint(test_df["person_id"]), "people in both train and test"

pd.DataFrame.from_dict(orient="index", data={
    name: {
        "people": len(df),
        "diabetic": df[TARGET].sum(),
        "diabetic_rate": round(df[TARGET].mean(), 3),
        "female_share": round((df["gender"] == "female").mean(), 3),
    }
    for name, df in {"full": feature_table, "train": train_df, "test": test_df}.items()
})

,people,diabetic,diabetic_rate,female_share
full,4742,766,0.162,0.518
train,3793,613,0.162,0.514
test,949,153,0.161,0.531


### Step 2 summary

- **Cleaned:** storage glitches fixed, unmeasurable diastolic readings set to missing, `taking_bp_meds` recoded to 0/1, `gender` and `age_years` given proper types.
- **Checked:** every sanity rule passes.
- **Split:** `train_df` (80%) and `test_df` (20%), stratified on `diabetes`, with no overlap.

**From here on, only `train_df` is used.** `test_df` stays sealed until Step 9.

**Next, Step 3 (EDA):** explore `train_df`: target balance, missing values, distributions and outliers, each feature against the target, correlations, and the effect of blood pressure medication.